# 00. Preparar bases — bronze Censo + CPF

Importa **censo_pessoas** + CPF bronze, filtra UF, infere nome da mãe **só no subset UF**, enriquece CEP via `data_cep_uniq.csv`, empilha `registro_unificado` e gera ground truth.

`REBUILD=False` reutiliza tabelas já materializadas no DuckDB.


In [ ]:
import sys
from pathlib import Path

PROB_DIR = Path.cwd()
if PROB_DIR.name == 'notebooks':
    PROB_DIR = PROB_DIR.parent
if str(PROB_DIR) not in sys.path:
    sys.path.insert(0, str(PROB_DIR))

%load_ext autoreload
%autoreload 2

REBUILD = False       # True = apaga e reconstrói tudo
REFILTER_GEO = True   # True = refaz filtro UF/município (sem rebuild completo)

import config
from config import (
    CENSO_CEP_ARQUIVO, CENSO_PESSOAS_ARQUIVO, CPF_ARQUIVO,
    OUTPUT_DIR, USE_PHONETIC_STRIP_VOWELS,
    CENSO_COL_ID_DOMICILIO, CENSO_COL_ID_MORADOR, CENSO_COL_PRIMEIRO_NOME,
    CENSO_COL_SEXO, CENSO_COL_SOBRENOME, CPF_COL_CEP, CPF_COL_CPF,
    CPF_COL_DATA_NASC, CPF_COL_NOME, CPF_COL_NOME_MAE, CPF_COL_SEXO,
    benchmark_checkpoint, censo_cep_join_on, censo_dob_sql, censo_municipio_expr,
    cep_norm_sql, cpf_municipio_expr, cpf_norm_sql, cpf_uf_expr, censo_uf_expr,
    export_parquet, geo_filter_clause, get_connection, idade_censo_sql, idade_cpf_sql,
    list_tables, materialize_censo_cep_lookup, normalize_date_sql,
    print_paths, require_input, require_tables,
)
from features import (
    NOME_MAE_COLUMNS,
    PESSOA_COLUMNS,
    clean_name_sql,
    name_feature_columns_sql,
    normalize_date_compacta_sql,
    normalize_sexo_sql,
    select_list_sql,
)
from inferir_pais import inferir_nome_mae_duckdb

# O filtro geográfico vem de config.py. Para sobrescrever só nesta sessão,
# descomente abaixo — reatribuir FILTRO_UF / FILTRO_MUNICIPIO aqui criaria
# apenas uma cópia local, sem efeito no filtro real.
# config.set_filtros(municipio=2111300)

print_paths()
for label, p in [
    ('CPF', CPF_ARQUIVO), ('CENSO_PESSOAS', CENSO_PESSOAS_ARQUIVO),
    ('CENSO_CEP', CENSO_CEP_ARQUIVO),
]:
    require_input(p, label=label)

con = get_connection()
print(
    'REBUILD:', REBUILD, '| REFILTER_GEO:', REFILTER_GEO,
    '| FILTRO_UF:', config.FILTRO_UF, '| FILTRO_MUNICIPIO:', config.FILTRO_MUNICIPIO,
)

if not REBUILD and 'registro_unificado' in list_tables(con):
    require_tables(con, ['registro_unificado'], notebook_origem='00')
    print('Tabelas finais já existem — defina REBUILD=True para refazer.')
else:
    print('Prosseguir com pipeline completo nas células abaixo.')


## 1. Inspecionar bronze


In [ ]:
if REBUILD:
    for label, path in [
        ('cpf', CPF_ARQUIVO),
        ('censo_pessoas', CENSO_PESSOAS_ARQUIVO),
    ]:
        print(f'\n=== {label} ===')
        display(con.execute(f"DESCRIBE SELECT * FROM read_parquet('{path}') LIMIT 0").df())


## 2. Importar bronze


In [ ]:
if REBUILD:
    for tbl in [
        'cpf_bronze_raw', 'censo_pessoas_raw',
        'cpf_filtrado', 'censo_pessoas_filtrado',
        'censo_pais_inferidos', 'censo_cep_lookup', 'censo_morador_cep',
        'cpf_staging', 'cpf_registros',
        'censo_staging', 'censo_registros',
        'registro_unificado',
    ]:
        con.execute(f'DROP TABLE IF EXISTS {tbl}')

    con.execute(f"CREATE OR REPLACE TABLE cpf_bronze_raw AS SELECT * FROM read_parquet('{CPF_ARQUIVO}')")
    con.execute(f"CREATE OR REPLACE TABLE censo_pessoas_raw AS SELECT * FROM read_parquet('{CENSO_PESSOAS_ARQUIVO}')")

    for t in ['cpf_bronze_raw', 'censo_pessoas_raw']:
        benchmark_checkpoint(con, t, f'SELECT COUNT(*) FROM {t}')


## 3. Filtrar por UF / município


In [ ]:
# Diagnóstico: os dois lados precisam gerar código IBGE de 7 dígitos.
# Se COD_UFMUN vier com 6 dígitos, o lpad produz 0XXXXXX e o filtro erra.
if 'cpf_bronze_raw' in list_tables(con):
    display(con.execute(f'''
    SELECT length(regexp_replace(CAST("{config.CPF_COL_UF}" AS VARCHAR), '[^0-9]', '', 'g')) AS n_digitos,
           COUNT(*) AS n
    FROM cpf_bronze_raw
    GROUP BY 1 ORDER BY 2 DESC
    ''').df())
    display(con.execute(f'''
    SELECT {cpf_municipio_expr('c')} AS cod_municipio_cpf, COUNT(*) AS n
    FROM cpf_bronze_raw c GROUP BY 1 ORDER BY 2 DESC LIMIT 5
    ''').df())

if 'censo_pessoas_raw' in list_tables(con):
    display(con.execute(f'''
    SELECT {censo_municipio_expr('p')} AS cod_municipio_censo, COUNT(*) AS n
    FROM censo_pessoas_raw p GROUP BY 1 ORDER BY 2 DESC LIMIT 5
    ''').df())

In [ ]:
if REBUILD or REFILTER_GEO:
    CPF_UF = cpf_uf_expr('c')
    CENSO_UF = censo_uf_expr('p')
    CPF_MUN = cpf_municipio_expr('c')
    CENSO_MUN = censo_municipio_expr('p')
    cpf_where = geo_filter_clause(CPF_UF, CPF_MUN)
    censo_where = geo_filter_clause(CENSO_UF, CENSO_MUN)
    filtro_ativo = config.FILTRO_UF is not None or config.FILTRO_MUNICIPIO is not None
    if filtro_ativo and cpf_where == 'TRUE' and censo_where == 'TRUE':
        raise RuntimeError(
            'Filtro configurado mas nenhuma cláusula gerada — use config.set_filtros().'
        )
    print('FILTRO_UF:', config.FILTRO_UF, '| FILTRO_MUNICIPIO:', config.FILTRO_MUNICIPIO)
    print('CPF WHERE:', cpf_where)
    print('CENSO WHERE:', censo_where)

    if 'cpf_bronze_raw' in list_tables(con):
        n_antes = con.execute('SELECT COUNT(*) FROM cpf_bronze_raw').fetchone()[0]
    else:
        n_antes = None

    con.execute(f'''
    CREATE OR REPLACE TABLE cpf_filtrado AS
    SELECT c.* FROM cpf_bronze_raw c
    WHERE {cpf_where}
    ''')

    con.execute(f'''
    CREATE OR REPLACE TABLE censo_pessoas_filtrado AS
    SELECT p.* FROM censo_pessoas_raw p
    WHERE {censo_where}
    ''')

    for t in ['cpf_filtrado', 'censo_pessoas_filtrado']:
        benchmark_checkpoint(con, t, f'SELECT COUNT(*) FROM {t}')
    if n_antes is not None:
        n_depois = con.execute('SELECT COUNT(*) FROM cpf_filtrado').fetchone()[0]
        print(f'CPF: {n_antes:,} bronze → {n_depois:,} filtrado')
        if filtro_ativo and n_depois == n_antes:
            raise RuntimeError(
                f'Filtro ativo mas nada foi filtrado ({n_depois:,} = bronze). '
                'Confira o formato de COD_UFMUN na célula de diagnóstico.'
            )
        if filtro_ativo and n_depois == 0:
            raise RuntimeError(
                'Filtro ativo e resultado vazio — provável divergência de formato '
                'entre COD_UFMUN (CPF) e o prefixo do setor censitário.'
            )
elif not REBUILD:
    print('Filtro geográfico pulado — defina REFILTER_GEO=True ou REBUILD=True')


## 3b. Diagnóstico da idade

Independente de `REBUILD`: roda sobre as tabelas filtradas e mostra onde a
idade se perde de cada lado. No Censo, `PECP0003` (1 ano ou mais) e `PECP0030`
(menos de 1 ano) dividem a população, então `ambos_nulos` alto é o sintoma a
procurar. No CPF a idade é derivada da data de nascimento, e aí a suspeita é o
formato de `DAT_NASCIMENTO` não bater com o que o `normalize_date_sql` espera.

In [ ]:
tabelas = list_tables(con)
ANOS, MESES = config.CENSO_COL_IDADE_ANOS, config.CENSO_COL_IDADE_MESES

if 'censo_pessoas_filtrado' in tabelas:
    print('=== CENSO: tipos das colunas de idade ===')
    display(con.execute(f'''
    DESCRIBE SELECT p."{ANOS}" AS {ANOS}, p."{MESES}" AS {MESES}
    FROM censo_pessoas_filtrado p LIMIT 0
    ''').df())

    IDADE_C = config.idade_censo_sql('p')
    display(con.execute(f'''
    SELECT
        COUNT(*) AS n,
        SUM(CASE WHEN p."{ANOS}" IS NULL THEN 1 ELSE 0 END) AS anos_nulo,
        SUM(CASE WHEN p."{MESES}" IS NULL THEN 1 ELSE 0 END) AS meses_nulo,
        SUM(CASE WHEN p."{ANOS}" IS NULL AND p."{MESES}" IS NULL
                 THEN 1 ELSE 0 END) AS ambos_nulos,
        SUM(CASE WHEN {IDADE_C} IS NULL THEN 1 ELSE 0 END) AS idade_nula,
        ROUND(100.0 * SUM(CASE WHEN {IDADE_C} IS NULL THEN 1 ELSE 0 END)
              / COUNT(*), 2) AS pct_idade_nula
    FROM censo_pessoas_filtrado p
    ''').df())

    # Valores crus: revela codificação inesperada (branco, '999', texto).
    for col in [ANOS, MESES]:
        print(f'--- {col}: 12 valores crus mais frequentes ---')
        display(con.execute(f'''
        SELECT CAST(p."{col}" AS VARCHAR) AS valor, COUNT(*) AS n
        FROM censo_pessoas_filtrado p GROUP BY 1 ORDER BY n DESC LIMIT 12
        ''').df())

if 'cpf_filtrado' in tabelas:
    print('\n=== CPF: da data de nascimento até a idade ===')
    DT_BASE = normalize_date_sql(f'c."{CPF_COL_DATA_NASC}"')
    DT = normalize_date_compacta_sql(f'c."{CPF_COL_DATA_NASC}"')
    display(con.execute(f'''
    DESCRIBE SELECT c."{CPF_COL_DATA_NASC}" AS {CPF_COL_DATA_NASC}
    FROM cpf_filtrado c LIMIT 0
    ''').df())
    # 'recuperado_compacto' > 0 significa que a base guarda YYYYMMDD e que era
    # isso que zerava a idade do CPF.
    display(con.execute(f'''
    SELECT
        COUNT(*) AS n,
        SUM(CASE WHEN c."{CPF_COL_DATA_NASC}" IS NULL THEN 1 ELSE 0 END) AS raw_nulo,
        SUM(CASE WHEN {DT_BASE} = '' THEN 1 ELSE 0 END) AS nao_parseou_antes,
        SUM(CASE WHEN {DT} = '' THEN 1 ELSE 0 END) AS nao_parseou_agora,
        SUM(CASE WHEN {DT_BASE} = '' AND {DT} <> '' THEN 1 ELSE 0 END)
            AS recuperado_compacto,
        SUM(CASE WHEN {idade_cpf_sql(DT)} IS NULL THEN 1 ELSE 0 END) AS idade_nula,
        ROUND(100.0 * SUM(CASE WHEN {idade_cpf_sql(DT)} IS NULL THEN 1 ELSE 0 END)
              / COUNT(*), 2) AS pct_idade_nula
    FROM cpf_filtrado c
    ''').df())
    print('--- DAT_NASCIMENTO: 12 valores crus mais frequentes ---')
    display(con.execute(f'''
    SELECT CAST(c."{CPF_COL_DATA_NASC}" AS VARCHAR) AS valor, COUNT(*) AS n
    FROM cpf_filtrado c GROUP BY 1 ORDER BY n DESC LIMIT 12
    ''').df())

## 4. Inferir nome da mãe (Censo — **após filtro UF**)

Usa `censo_pessoas_filtrado` — não roda na base nacional inteira.


In [ ]:
if REBUILD:
    inferir_nome_mae_duckdb(con, source_table='censo_pessoas_filtrado')
    benchmark_checkpoint(con, 'censo_pais_inferidos', 'SELECT COUNT(*) FROM censo_pais_inferidos')
    con.execute('''
    SELECT
        COUNT(*) AS n,
        SUM(CASE WHEN nome_mae IS NOT NULL AND TRIM(nome_mae) <> '' THEN 1 ELSE 0 END) AS com_mae,
        ROUND(100.0 * SUM(CASE WHEN nome_mae IS NOT NULL AND TRIM(nome_mae) <> '' THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_mae
    FROM censo_pais_inferidos
    ''').df()


## 5. CEP Censo (`data_cep_uniq.csv` — **após filtro UF**)

LEFT JOIN em `censo_pessoas_filtrado` por `B0000`↔`COD_SETOR` (15 díg.), `NUM_QUADRA`, `NUM_FACE`.

Lookup de CEP filtrado por UF quando `FILTRO_UF` está definido.


In [ ]:
if REBUILD:
    materialize_censo_cep_lookup(con)
    benchmark_checkpoint(con, 'censo_cep_lookup', 'SELECT COUNT(*) FROM censo_cep_lookup')

    join_on = censo_cep_join_on('p', 'k')
    con.execute(f'''
    CREATE OR REPLACE TABLE censo_morador_cep AS
    SELECT
        CAST(p.{CENSO_COL_ID_MORADOR} AS VARCHAR) AS person_id_censo,
        COALESCE(k.cep, '') AS cep
    FROM censo_pessoas_filtrado p
    LEFT JOIN censo_cep_lookup k ON {join_on}
    ''')

    con.execute('''
    SELECT
        COUNT(*) AS n,
        SUM(CASE WHEN cep <> '' THEN 1 ELSE 0 END) AS com_cep,
        ROUND(100.0 * SUM(CASE WHEN cep <> '' THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_cep
    FROM censo_morador_cep
    ''').df()


## 6. CPF — limpeza e nomes


In [ ]:
if REBUILD:
    CPF_N = cpf_norm_sql(f'c."{CPF_COL_CPF}"')
    DT_NASC = normalize_date_compacta_sql(f'c."{CPF_COL_DATA_NASC}"')
    NOME_MAE = f'c."{CPF_COL_NOME_MAE}"'
    SEXO = f'c."{CPF_COL_SEXO}"'
    CEP = cep_norm_sql(f'c."{CPF_COL_CEP}"')
    UF_COL = cpf_uf_expr('c')
    MUN_COL = cpf_municipio_expr('c')
    SEXO_N = normalize_sexo_sql('sexo_raw')

    # ANO_OBITO existe no bronze do CPF. Falha aqui é melhor que seguir com a
    # coluna nula, que desligaria o filtro de óbito do NB00b sem avisar.
    cpf_cols = set(con.execute('SELECT * FROM cpf_filtrado LIMIT 0').df().columns)
    if config.CPF_COL_ANO_OBITO not in cpf_cols:
        candidatas = sorted(c for c in cpf_cols if 'OBITO' in c.upper())
        raise KeyError(
            f'{config.CPF_COL_ANO_OBITO} não existe no CPF bronze. '
            f'Candidatas: {candidatas or "nenhuma"}. '
            'Ajuste CPF_COL_ANO_OBITO em config.py.'
        )
    ANO_OBITO = f'TRY_CAST(c."{config.CPF_COL_ANO_OBITO}" AS INTEGER)'

    con.execute(f'''
    CREATE OR REPLACE TABLE cpf_staging AS
    SELECT
        {CPF_N} AS cpf_norm,
        TRIM(CAST(c."{CPF_COL_NOME}" AS VARCHAR)) AS nome_completo_raw,
        {DT_NASC} AS data_nascimento,
        {idade_cpf_sql(DT_NASC)} AS idade,
        CAST({NOME_MAE} AS VARCHAR) AS nome_mae_raw,
        CAST({SEXO} AS VARCHAR) AS sexo_raw,
        {CEP} AS cep,
        {UF_COL} AS uf,
        {MUN_COL} AS cod_municipio,
        {ANO_OBITO} AS ano_obito
    FROM cpf_filtrado c
    WHERE {CPF_N} IS NOT NULL
    ''')

    # Featurização em SQL: um passe sobre o staging gera pessoa e mãe.
    # NULLIF nas colunas da mãe: a expressão devolve '' onde antes vinha NULL,
    # e string vazia faria o Splink tratar '' = '' como match de verdade.
    PESSOA = name_feature_columns_sql('nome_completo_norm', col_map=PESSOA_COLUMNS)
    MAE = {
        alias: f"NULLIF({expr}, '')"
        for alias, expr in name_feature_columns_sql(
            'nome_mae_norm', col_map=NOME_MAE_COLUMNS
        ).items()
    }

    con.execute(f'''
    CREATE OR REPLACE TABLE cpf_registros AS
    WITH norm AS (
        SELECT *,
            {clean_name_sql('nome_completo_raw')} AS nome_completo_norm,
            {clean_name_sql('nome_mae_raw')} AS nome_mae_norm
        FROM cpf_staging
    )
    SELECT
        'cpf_' || cpf_norm AS unique_id, 'cpf' AS origem, cpf_norm,
        {select_list_sql(PESSOA)},
        data_nascimento,
        {select_list_sql(MAE)},
        {SEXO_N} AS sexo,
        idade,
        cep, CAST(uf AS VARCHAR) AS uf,
        CAST(cod_municipio AS VARCHAR) AS cod_municipio,
        ano_obito,
        CAST(NULL AS VARCHAR) AS person_id_censo,
        CAST(NULL AS VARCHAR) AS id_domicilio
    FROM norm
    ''')
    benchmark_checkpoint(con, 'cpf_registros', 'SELECT COUNT(*) FROM cpf_registros')


## 7. Censo — limpeza e nomes


In [ ]:
if REBUILD:
    DT_PESSOA = censo_dob_sql()
    DT_NASC_C = normalize_date_sql(DT_PESSOA)
    NOME_COMPLETO = f"TRIM(COALESCE(CAST(p.{CENSO_COL_PRIMEIRO_NOME} AS VARCHAR), '') || ' ' || COALESCE(CAST(p.{CENSO_COL_SOBRENOME} AS VARCHAR), ''))"
    UF_C = censo_uf_expr('p')
    MUN_C = censo_municipio_expr('p')
    SEXO_N = normalize_sexo_sql('sexo_raw')

    con.execute(f'''
    CREATE OR REPLACE TABLE censo_staging AS
    SELECT
        CAST(p.{CENSO_COL_ID_MORADOR} AS VARCHAR) AS person_id_censo,
        CAST(p.{CENSO_COL_ID_DOMICILIO} AS VARCHAR) AS id_domicilio,
        {NOME_COMPLETO} AS nome_completo_raw,
        {DT_NASC_C} AS data_nascimento,
        {idade_censo_sql('p')} AS idade,
        CAST(p.{CENSO_COL_SEXO} AS VARCHAR) AS sexo_raw,
        {UF_C} AS uf,
        {MUN_C} AS cod_municipio,
        COALESCE(e.cep, '') AS cep,
        m.nome_mae AS nome_mae_inferido
    FROM censo_pessoas_filtrado p
    LEFT JOIN censo_morador_cep e ON CAST(p.{CENSO_COL_ID_MORADOR} AS VARCHAR) = e.person_id_censo
    LEFT JOIN censo_pais_inferidos m ON CAST(p.{CENSO_COL_ID_MORADOR} AS VARCHAR) = m.person_id_censo
    ''')

    PESSOA = name_feature_columns_sql('nome_completo_norm', col_map=PESSOA_COLUMNS)
    MAE = {
        alias: f"NULLIF({expr}, '')"
        for alias, expr in name_feature_columns_sql(
            'nome_mae_norm', col_map=NOME_MAE_COLUMNS
        ).items()
    }

    con.execute(f'''
    CREATE OR REPLACE TABLE censo_registros AS
    WITH norm AS (
        SELECT *,
            {clean_name_sql('nome_completo_raw')} AS nome_completo_norm,
            {clean_name_sql('nome_mae_inferido')} AS nome_mae_norm
        FROM censo_staging
        WHERE person_id_censo IS NOT NULL
    )
    SELECT
        'censo_' || person_id_censo AS unique_id, 'censo' AS origem,
        CAST(NULL AS VARCHAR) AS cpf_norm,
        {select_list_sql(PESSOA)},
        data_nascimento,
        {select_list_sql(MAE)},
        {SEXO_N} AS sexo,
        idade,
        cep, CAST(uf AS VARCHAR) AS uf,
        CAST(cod_municipio AS VARCHAR) AS cod_municipio,
        CAST(NULL AS INTEGER) AS ano_obito,
        person_id_censo, id_domicilio
    FROM norm
    ''')
    benchmark_checkpoint(con, 'censo_registros', 'SELECT COUNT(*) FROM censo_registros')


## 8. Empilhar

Ground truth não entra aqui: a coorte só é carregada na validação (NB03).


In [ ]:
if REBUILD:
    phon_cols = (
        'nome_completo_phon_sv, primeiro_nome_phon_sv, nome_meio_phon_sv, ultimo_nome_phon_sv,'
        'nome_mae_phon_sv, primeiro_nome_mae_phon_sv, nome_meio_mae_phon_sv, ultimo_nome_mae_phon_sv,'
        if USE_PHONETIC_STRIP_VOWELS else ''
    )
    cols = '''
        unique_id, origem, cpf_norm, nome_completo, primeiro_nome, nome_meio, ultimo_nome,
        nome_completo_phon, primeiro_nome_phon, nome_meio_phon, ultimo_nome_phon,
        nome_mae, primeiro_nome_mae, nome_meio_mae, ultimo_nome_mae,
        nome_mae_phon, primeiro_nome_mae_phon, nome_meio_mae_phon, ultimo_nome_mae_phon,
    '''
    tail = (
        'data_nascimento, sexo, idade, cep, uf, cod_municipio, ano_obito, '
        'person_id_censo, id_domicilio'
    )

    con.execute(f'''
    CREATE OR REPLACE TABLE registro_unificado AS
    SELECT {cols} {phon_cols} {tail}
    FROM cpf_registros
    UNION ALL
    SELECT {cols} {phon_cols} {tail}
    FROM censo_registros
    ''')

    benchmark_checkpoint(con, 'registro_unificado', 'SELECT COUNT(*) FROM registro_unificado')


## 9. Export


In [ ]:
if REBUILD:
    p1 = export_parquet(con, 'registro_unificado', path=OUTPUT_DIR / 'registro_unificado.parquet')
    print('Exportado:', p1)
con.close()
